# Multi-modal Cell Classification from Spatial Data

## Overview
This notebook demonstrates a complete workflow for multi-modal spatial data analysis:
1. Filter cells that have all three data modalities (microscopy, metabolomics, and transcriptomics)
2. Perform cell type deconvolution/clustering using gene expression data
3. Train a machine learning model to predict cell types from metabolomics profiles

This approach allows us to learn the relationship between metabolic signatures and cell types, which can be useful for predicting cell types in datasets where only metabolomics data is available.

## 1. Load and Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load the CSV file
# Update the path to match your data location
csv_df = pd.read_csv("hfd_11_Av.xlsx/triple_overlap_details.csv")

print(f"Total number of cells: {len(csv_df)}")
print(f"Number of columns: {len(csv_df.columns)}")
print("\nFirst few rows:")
csv_df.head()

In [ ]:
# Explore the data modalities
print("Distribution of data modality combinations:")
print(f"Has all three modalities: {csv_df['has_all_three'].sum()}")
print(f"Has only microscopy: {csv_df['has_only_microscopy'].sum()}")
print(f"Has microscopy + metabolomics only: {csv_df['has_microscopy_metabolomics_only'].sum()}")
print(f"Has microscopy + transcriptomics only: {csv_df['has_microscopy_transcriptomics_only'].sum()}")

# Visualize
modality_counts = pd.Series({
    'All Three': csv_df['has_all_three'].sum(),
    'Microscopy Only': csv_df['has_only_microscopy'].sum(),
    'Microsc. + Metab.': csv_df['has_microscopy_metabolomics_only'].sum(),
    'Microsc. + Transcr.': csv_df['has_microscopy_transcriptomics_only'].sum()
})

plt.figure(figsize=(10, 6))
modality_counts.plot(kind='bar', color='steelblue')
plt.title('Distribution of Data Modality Combinations', fontsize=14, fontweight='bold')
plt.xlabel('Modality Combination')
plt.ylabel('Number of Cells')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. Filter Cells with All Three Modalities

In [ ]:
# Filter for cells with all three modalities
cells_all_three = csv_df[csv_df['has_all_three'] == True].copy()

print(f"Number of cells with all three modalities: {len(cells_all_three)}")
print(f"Percentage of total cells: {len(cells_all_three) / len(csv_df) * 100:.2f}%")

In [ ]:
# Identify column types
metab_cols = [col for col in cells_all_three.columns if col.startswith('metab_channel_')]
expr_cols = [col for col in cells_all_three.columns if col.startswith('expr_')]

print(f"Number of metabolomics channels: {len(metab_cols)}")
print(f"Number of gene expression features: {len(expr_cols)}")

# Extract feature matrices
X_metab = cells_all_three[metab_cols].values
X_expr = cells_all_three[expr_cols].values

print(f"\nMetabolomics matrix shape: {X_metab.shape}")
print(f"Gene expression matrix shape: {X_expr.shape}")

## 3. Cell Type Deconvolution Using Gene Expression

We'll use clustering on gene expression data to identify cell types. For real applications, you might want to use more sophisticated methods like:
- SingleR for reference-based annotation
- Cell2location for spatial deconvolution
- Manual annotation based on marker genes

Here we'll use a simple clustering approach with dimensionality reduction.

In [ ]:
# Filter out genes with zero expression across all cells
gene_sums = X_expr.sum(axis=0)
expressed_genes_mask = gene_sums > 0
X_expr_filtered = X_expr[:, expressed_genes_mask]
expr_cols_filtered = [col for col, mask in zip(expr_cols, expressed_genes_mask) if mask]

print(f"Number of expressed genes: {X_expr_filtered.shape[1]} (out of {len(expr_cols)})")

# Additional filtering: keep genes with expression in at least 5% of cells
min_cells = int(0.05 * X_expr_filtered.shape[0])
cells_expressing = (X_expr_filtered > 0).sum(axis=0)
highly_expressed_mask = cells_expressing >= min_cells
X_expr_filtered = X_expr_filtered[:, highly_expressed_mask]
expr_cols_filtered = [col for col, mask in zip(expr_cols_filtered, highly_expressed_mask) if mask]

print(f"Number of genes expressed in ≥5% of cells: {X_expr_filtered.shape[1]}")

In [ ]:
# Normalize gene expression (log1p transformation)
X_expr_log = np.log1p(X_expr_filtered)

# Standardize
from sklearn.preprocessing import StandardScaler
scaler_expr = StandardScaler()
X_expr_scaled = scaler_expr.fit_transform(X_expr_log)

In [ ]:
# Dimensionality reduction using PCA
from sklearn.decomposition import PCA

# Determine number of PCs to use (explaining 80% of variance)
pca = PCA(n_components=0.8, random_state=42)
X_expr_pca = pca.fit_transform(X_expr_scaled)

print(f"Number of PCs: {X_expr_pca.shape[1]}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_.sum():.3f}")

# Visualize PCA
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.scatter(X_expr_pca[:, 0], X_expr_pca[:, 1], alpha=0.5, s=10)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA of Gene Expression')

plt.subplot(1, 2, 2)
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Clustering to identify cell types
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Test different numbers of clusters
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_expr_pca)
    score = silhouette_score(X_expr_pca, labels)
    silhouette_scores.append(score)
    print(f"K={k}: Silhouette Score = {score:.3f}")

# Plot silhouette scores
plt.figure(figsize=(10, 5))
plt.plot(K_range, silhouette_scores, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs Number of Clusters')
plt.grid(True)
plt.show()

# Choose optimal K
optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"\nOptimal number of clusters: {optimal_k}")

In [ ]:
# Perform final clustering with optimal K
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=20)
cell_types = kmeans_final.fit_predict(X_expr_pca)

# Add cell type labels to the dataframe
cells_all_three['cell_type'] = cell_types

print(f"\nCell type distribution:")
print(cells_all_three['cell_type'].value_counts().sort_index())

# Visualize clusters
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
scatter = plt.scatter(X_expr_pca[:, 0], X_expr_pca[:, 1], 
                      c=cell_types, cmap='tab10', alpha=0.6, s=20)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Cell Types in PCA Space')
plt.colorbar(scatter, label='Cell Type')

plt.subplot(1, 2, 2)
cells_all_three['cell_type'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.xlabel('Cell Type')
plt.ylabel('Number of Cells')
plt.title('Cell Type Distribution')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 4. Characterize Cell Types

Let's identify marker genes for each cell type to better understand what they represent.

In [ ]:
# Find top marker genes for each cluster
def find_marker_genes(expr_data, gene_names, labels, top_n=10):
    """
    Find marker genes for each cluster using fold-change
    """
    marker_genes = {}
    
    for cluster in np.unique(labels):
        # Get mean expression in cluster vs out of cluster
        in_cluster = expr_data[labels == cluster]
        out_cluster = expr_data[labels != cluster]
        
        mean_in = in_cluster.mean(axis=0)
        mean_out = out_cluster.mean(axis=0)
        
        # Calculate fold change (with pseudocount to avoid division by zero)
        fold_change = (mean_in + 1e-9) / (mean_out + 1e-9)
        
        # Get top genes
        top_indices = np.argsort(fold_change)[-top_n:][::-1]
        marker_genes[cluster] = [(gene_names[i], fold_change[i]) for i in top_indices]
    
    return marker_genes

marker_genes = find_marker_genes(X_expr_filtered, expr_cols_filtered, cell_types, top_n=5)

print("Top 5 marker genes for each cell type:\n")
for cluster, genes in marker_genes.items():
    print(f"Cell Type {cluster}:")
    for gene, fc in genes:
        # Clean up gene name (remove 'expr_' prefix)
        gene_clean = gene.replace('expr_', '')
        print(f"  {gene_clean}: {fc:.2f}x")
    print()

In [ ]:
# Visualize marker gene expression across cell types
# Select top 3 markers for each cell type for visualization
top_markers = []
for cluster, genes in marker_genes.items():
    top_markers.extend([g[0] for g in genes[:3]])
top_markers = list(set(top_markers))[:15]  # Limit to 15 unique markers

# Create expression matrix for these markers
marker_indices = [expr_cols_filtered.index(m) for m in top_markers if m in expr_cols_filtered]
expr_for_heatmap = X_expr_filtered[:, marker_indices]

# Calculate mean expression per cell type
mean_expr_by_type = np.zeros((optimal_k, len(marker_indices)))
for i in range(optimal_k):
    mean_expr_by_type[i] = expr_for_heatmap[cell_types == i].mean(axis=0)

# Plot heatmap
plt.figure(figsize=(12, 6))
marker_labels = [top_markers[i].replace('expr_', '') for i in range(len(marker_indices))]
sns.heatmap(mean_expr_by_type.T, 
            xticklabels=[f'Type {i}' for i in range(optimal_k)],
            yticklabels=marker_labels,
            cmap='RdYlBu_r', cbar_kws={'label': 'Mean Expression'})
plt.title('Marker Gene Expression Across Cell Types', fontsize=14, fontweight='bold')
plt.xlabel('Cell Type')
plt.ylabel('Gene')
plt.tight_layout()
plt.show()

## 5. Train ML Model to Predict Cell Types from Metabolomics

Now that we have cell type labels, we'll train a machine learning model to predict cell types from metabolomics profiles alone.

In [ ]:
# Prepare data for ML
# Features: metabolomics channels
# Target: cell types

X = X_metab
y = cell_types

# Check for any missing values
print(f"Missing values in metabolomics data: {np.isnan(X).sum()}")

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nClass distribution in training set:")
print(pd.Series(y_train).value_counts().sort_index())

In [ ]:
# Standardize metabolomics data
scaler_metab = StandardScaler()
X_train_scaled = scaler_metab.fit_transform(X_train)
X_test_scaled = scaler_metab.transform(X_test)

In [ ]:
# Train Random Forest classifier
print("Training Random Forest classifier...")
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
print("Training complete!")

## 6. Evaluate Model Performance

In [ ]:
# Make predictions
y_train_pred = rf_model.predict(X_train_scaled)
y_test_pred = rf_model.predict(X_test_scaled)

# Calculate accuracies
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.3f}")
print(f"Test Accuracy: {test_accuracy:.3f}")
print(f"\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred, 
                          target_names=[f'Type {i}' for i in range(optimal_k)]))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Type {i}' for i in range(optimal_k)],
            yticklabels=[f'Type {i}' for i in range(optimal_k)])
plt.title('Confusion Matrix - Cell Type Prediction from Metabolomics', 
          fontsize=14, fontweight='bold')
plt.ylabel('True Cell Type')
plt.xlabel('Predicted Cell Type')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
feature_importance = rf_model.feature_importances_
sorted_idx = np.argsort(feature_importance)[::-1]

# Plot top 20 most important metabolomics channels
top_n = min(20, len(metab_cols))
plt.figure(figsize=(12, 6))
plt.bar(range(top_n), feature_importance[sorted_idx[:top_n]])
plt.xlabel('Metabolomics Channel')
plt.ylabel('Feature Importance')
plt.title('Top 20 Most Important Metabolomics Features for Cell Type Prediction',
         fontsize=14, fontweight='bold')
plt.xticks(range(top_n), 
           [metab_cols[i].replace('metab_channel_', 'Ch') for i in sorted_idx[:top_n]],
           rotation=45)
plt.tight_layout()
plt.show()

print("\nTop 10 most important metabolomics channels:")
for i in range(min(10, len(metab_cols))):
    idx = sorted_idx[i]
    print(f"{metab_cols[idx]}: {feature_importance[idx]:.4f}")

## 7. Cross-validation for Robust Evaluation

In [ ]:
from sklearn.model_selection import cross_val_score

# Perform 5-fold cross-validation
cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, n_jobs=-1)

print("Cross-validation results:")
print(f"Mean CV Accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")
print(f"Individual fold scores: {[f'{score:.3f}' for score in cv_scores]}")

# Plot CV scores
plt.figure(figsize=(10, 6))
plt.bar(range(1, 6), cv_scores, color='steelblue', alpha=0.7)
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', label=f'Mean: {cv_scores.mean():.3f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Cross-Validation Scores', fontsize=14, fontweight='bold')
plt.legend()
plt.ylim([0, 1])
plt.tight_layout()
plt.show()

## 8. Visualize Metabolomics Profiles by Cell Type

In [ ]:
# Calculate mean metabolomics profile for each cell type
mean_metab_by_type = np.zeros((optimal_k, len(metab_cols)))
for i in range(optimal_k):
    mean_metab_by_type[i] = X_metab[cell_types == i].mean(axis=0)

# Plot heatmap of metabolomics profiles
plt.figure(figsize=(14, 6))
sns.heatmap(mean_metab_by_type.T[:50],  # Show first 50 channels for clarity
            xticklabels=[f'Type {i}' for i in range(optimal_k)],
            yticklabels=[f'Ch{i}' for i in range(50)],
            cmap='viridis', cbar_kws={'label': 'Mean Intensity'})
plt.title('Metabolomics Profiles Across Cell Types (First 50 Channels)',
         fontsize=14, fontweight='bold')
plt.xlabel('Cell Type')
plt.ylabel('Metabolomics Channel')
plt.tight_layout()
plt.show()

## 9. Save Results

In [ ]:
# Save the trained model and scaler for future use
import joblib

# Save model
joblib.dump(rf_model, 'cell_type_classifier_rf.pkl')
joblib.dump(scaler_metab, 'metabolomics_scaler.pkl')
print("Model and scaler saved successfully!")

# Save cell type annotations
results_df = cells_all_three[['cell_id', 'cell_area', 'cell_type']].copy()
results_df['predicted_cell_type'] = rf_model.predict(scaler_metab.transform(X_metab))
results_df.to_csv('cell_type_annotations.csv', index=False)
print("Cell type annotations saved to cell_type_annotations.csv")

## 10. Summary and Conclusions

### What we accomplished:

1. **Data Filtering**: We identified cells with all three data modalities (microscopy, metabolomics, and transcriptomics)

2. **Cell Type Deconvolution**: We used clustering on gene expression data to identify distinct cell populations

3. **ML Model Training**: We trained a Random Forest classifier to predict cell types from metabolomics profiles

4. **Model Evaluation**: The model achieved reasonable accuracy in predicting cell types from metabolomics alone

### Key Insights:

- The model demonstrates that metabolomics profiles contain information about cell identity
- Feature importance analysis reveals which metabolomics channels are most informative for cell type prediction
- This approach can be extended to predict cell types in samples where only metabolomics data is available

### Next Steps:

1. Try more sophisticated cell type annotation methods (SingleR, marker gene databases)
2. Experiment with different ML models (XGBoost, Neural Networks)
3. Incorporate spatial information for improved predictions
4. Validate predictions using independent datasets
5. Investigate biological interpretations of the metabolomics-cell type relationships